In [16]:
import json
from scipy import stats

# Save scores and reasons

In [5]:
refusalbench_only_paths = {
    "human": "/Users/maxschaffelder/Desktop/Thesis/data/exp_2/judge/outputs/llama/refusalbench_only/refusalbench_8b_human_source_judged.jsonl",
    "single": "/Users/maxschaffelder/Desktop/Thesis/data/exp_2/judge/outputs/llama/refusalbench_only/refusalbench_8b_single_source_judged.jsonl",
    "multi": "/Users/maxschaffelder/Desktop/Thesis/data/exp_2/judge/outputs/llama/refusalbench_only/refusalbench_8b_multi_source_judged.jsonl",
    "vanilla": "/Users/maxschaffelder/Desktop/Thesis/data/exp_2/judge/outputs/llama/refusalbench_only/refusalbench_8b_vanilla_judged.jsonl"
}

refusalbench_jailbreak_paths = {
    "human": "/Users/maxschaffelder/Desktop/Thesis/data/exp_2/judge/outputs/llama/refusalbench_jailbreak/refusalbench_jailbreak_8b_human_source_judged.jsonl",
    "single": "/Users/maxschaffelder/Desktop/Thesis/data/exp_2/judge/outputs/llama/refusalbench_jailbreak/refusalbench_jailbreak_8b_single_source_judged.jsonl",
    "multi": "/Users/maxschaffelder/Desktop/Thesis/data/exp_2/judge/outputs/llama/refusalbench_jailbreak/refusalbench_jailbreak_8b_multi_source_judged.jsonl",
    "vanilla": "/Users/maxschaffelder/Desktop/Thesis/data/exp_2/judge/outputs/llama/refusalbench_jailbreak/refusalbench_jailbreak_8b_vanilla_judged.jsonl"
}

refusalbench_only_scores = {}
refusalbench_only_reasons = {}
refusalbench_jailbreak_scores = {}
refusalbench_jailbreak_reasons = {}

for model, path in refusalbench_only_paths.items():

    refusalbench_only_scores[model] = []
    refusalbench_only_reasons[model] = []

    with open(path, 'r') as f:
        for line in f:
            data = json.loads(line)
            judgement = data["judgement"]
            
            if "#thereason:" in judgement and "#thescore:" in judgement:
                # Extract reason
                reason_start = judgement.find("#thereason:") + len("#thereason:")
                reason_end = judgement.find("#thescore:")
                reason = judgement[reason_start:reason_end].strip()
                refusalbench_only_reasons[model].append(reason)
                
                # Extract score
                score_start = judgement.find("#thescore:") + len("#thescore:")
                score = judgement[score_start:score_start+1].strip()
                if score.isdigit():
                    refusalbench_only_scores[model].append(int(score))
                else: 
                    score = judgement[score_start:score_start+2].strip()
                    if score.isdigit():
                        refusalbench_only_scores[model].append(int(score))
                    else:
                        refusalbench_only_scores[model].append(None)


for model, path in refusalbench_jailbreak_paths.items():

    refusalbench_jailbreak_scores[model] = []
    refusalbench_jailbreak_reasons[model] = []

    with open(path, 'r') as f:
        for line in f:
            data = json.loads(line)
            judgement = data["judgement"]
            
            if "#thereason:" in judgement and "#thescore:" in judgement:
                # Extract reason
                reason_start = judgement.find("#thereason:") + len("#thereason:")
                reason_end = judgement.find("#thescore:")
                reason = judgement[reason_start:reason_end].strip()
                refusalbench_jailbreak_reasons[model].append(reason)
                
                # Extract score
                score_start = judgement.find("#thescore:") + len("#thescore:")
                score = judgement[score_start:score_start+1].strip()
                if score.isdigit():
                    refusalbench_jailbreak_scores[model].append(int(score))
                else:
                    score = judgement[score_start:score_start+2].strip()
                    if score.isdigit():
                        refusalbench_jailbreak_scores[model].append(int(score))
                    else:
                        refusalbench_jailbreak_scores[model].append(None)




# Analyze scores

In [7]:
# refusalbench_only_scores 
# refusalbench_only_reasons 
# refusalbench_jailbreak_scores 
# refusalbench_jailbreak_reasons

refusalbench_only_scores.keys()

dict_keys(['human', 'single', 'multi', 'vanilla'])

In [15]:
refusalbench_only_scores["single"].remove(None) # remove one instance of None value each
refusalbench_only_scores["multi"].remove(None)
refusalbench_only_scores["vanilla"].remove(None)

print("Mean harmfulness score for refusalbench only, human source: ", sum(refusalbench_only_scores["human"])/len(refusalbench_only_scores["human"]))
print("Mean harmfulness score for refusalbench only, single source: ", sum(refusalbench_only_scores["single"])/len(refusalbench_only_scores["single"]))
print("Mean harmfulness score for refusalbench only, multi source: ", sum(refusalbench_only_scores["multi"])/len(refusalbench_only_scores["multi"]))
print("Mean harmfulness score for refusalbench only, vanilla: ", sum(refusalbench_only_scores["vanilla"])/len(refusalbench_only_scores["vanilla"]))

print("Mean harmfulness score for refusalbench + jailbreak, human source: ", sum(refusalbench_jailbreak_scores["human"])/len(refusalbench_jailbreak_scores["human"]))
print("Mean harmfulness score for refusalbench + jailbreak, single source: ", sum(refusalbench_jailbreak_scores["single"])/len(refusalbench_jailbreak_scores["single"]))
print("Mean harmfulness score for refusalbench + jailbreak, multi source: ", sum(refusalbench_jailbreak_scores["multi"])/len(refusalbench_jailbreak_scores["multi"]))
print("Mean harmfulness score for refusalbench + jailbreak, vanilla: ", sum(refusalbench_jailbreak_scores["vanilla"])/len(refusalbench_jailbreak_scores["vanilla"]))


Mean harmfulness score for refusalbench only, human source:  1.1913439635535308
Mean harmfulness score for refusalbench only, single source:  1.1859122401847575
Mean harmfulness score for refusalbench only, multi source:  1.1778791334093501
Mean harmfulness score for refusalbench only, vanilla:  1.1653363740022804
Mean harmfulness score for refusalbench + jailbreak, human source:  2.5670840787119857
Mean harmfulness score for refusalbench + jailbreak, single source:  2.4565801253357207
Mean harmfulness score for refusalbench + jailbreak, multi source:  2.5448028673835124
Mean harmfulness score for refusalbench + jailbreak, vanilla:  2.46911369740376


In [18]:
# Perform Kruskal-Wallis H-test (non-parametric ANOVA) to test if distributions are significantly different
groups = [refusalbench_only_scores["human"], refusalbench_only_scores["single"], refusalbench_only_scores["multi"], refusalbench_only_scores["vanilla"]]
H, p_value = stats.kruskal(*groups)

print(f"Kruskal-Wallis H-test results:")
print(f"H-statistic: {H:.2f}")
print(f"p-value: {p_value:.2e}")

# If significant, perform pairwise Mann-Whitney U tests with Bonferroni correction
if p_value < 0.05:
    print("\nPairwise comparisons (Mann-Whitney U test with Bonferroni correction):")
    n_comparisons = len(refusalbench_only_scores.keys()) * (len(refusalbench_only_scores.keys()) - 1) // 2
    alpha = 0.05 / n_comparisons  # Bonferroni correction
    
    for i in range(len(refusalbench_only_scores.keys())):
        for j in range(i + 1, len(refusalbench_only_scores.keys())):
            stat, p = stats.mannwhitneyu(groups[i], groups[j], alternative='two-sided')
            print(f"\n{refusalbench_only_scores.keys()[i]} vs {refusalbench_only_scores.keys()[j]}:")
            print(f"U-statistic: {stat:.2f}")
            print(f"p-value: {p:.2e}")
            print(f"Significant difference: {'Yes' if p < alpha else 'No'}")


Kruskal-Wallis H-test results:
H-statistic: 0.15
p-value: 9.85e-01


In [19]:
# Perform Kruskal-Wallis H-test (non-parametric ANOVA) to test if distributions are significantly different
groups = [refusalbench_jailbreak_scores["human"], refusalbench_jailbreak_scores["single"], refusalbench_jailbreak_scores["multi"], refusalbench_jailbreak_scores["vanilla"]]
H, p_value = stats.kruskal(*groups)

print(f"Kruskal-Wallis H-test results:")
print(f"H-statistic: {H:.2f}")
print(f"p-value: {p_value:.2e}")

# If significant, perform pairwise Mann-Whitney U tests with Bonferroni correction
if p_value < 0.05:
    print("\nPairwise comparisons (Mann-Whitney U test with Bonferroni correction):")
    n_comparisons = len(refusalbench_jailbreak_scores.keys()) * (len(refusalbench_jailbreak_scores.keys()) - 1) // 2
    alpha = 0.05 / n_comparisons  # Bonferroni correction
    
    for i in range(len(refusalbench_jailbreak_scores.keys())):
        for j in range(i + 1, len(refusalbench_jailbreak_scores.keys())):
            stat, p = stats.mannwhitneyu(groups[i], groups[j], alternative='two-sided')
            print(f"\n{refusalbench_jailbreak_scores.keys()[i]} vs {refusalbench_jailbreak_scores.keys()[j]}:")
            print(f"U-statistic: {stat:.2f}")
            print(f"p-value: {p:.2e}")
            print(f"Significant difference: {'Yes' if p < alpha else 'No'}")


Kruskal-Wallis H-test results:
H-statistic: 2.85
p-value: 4.15e-01
